<a href="https://colab.research.google.com/github/shashankshekhar9420-creator/Neural-Probabilistic-Language-Model/blob/main/PROJECT_NPLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
#import libraries
import random
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [8]:
#set random seed
SEED=42
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
  torch.cuda.manual_seed_all(SEED)

In [9]:
#select device
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [10]:
#define hyperparameters
#context window size: (n-1) previous words
CONTEXT_SIZE = 3 #Number of previous words used to predict the next word.
#embedding dimension (m in paper)
EMBEDDING_DIM = 50 #Size of each word's embedding vector.
#hidden layer size
HIDDEN_DIM = 128 #Number of neurons in the hidden layer.
#optimization
LEARNING_RATE = 0.1
BATCH_SIZE = 64
EPOCHS = 20
#vocabulary
MIN_FREQ = 1 #Minimum frequency a word must have to be included in the vocabulary.
UNK_TOKEN = '<UNK>' #Special token used to represent rare or unseen words.

In [11]:
print("Configuration")
print("-" * 30)
print(f"Device          : {device}")
print(f"Context Size    : {CONTEXT_SIZE}")
print(f"Embedding Dim   : {EMBEDDING_DIM}")
print(f"Hidden Dim      : {HIDDEN_DIM}")
print(f"Batch Size      : {BATCH_SIZE}")
print(f"Learning Rate   : {LEARNING_RATE}")
print(f"Epochs          : {EPOCHS}")

Configuration
------------------------------
Device          : cuda
Context Size    : 3
Embedding Dim   : 50
Hidden Dim      : 128
Batch Size      : 64
Learning Rate   : 0.1
Epochs          : 20


In [12]:
# Path to the corpus
CORPUS_PATH = "tiny_shakespeare.txt"

# Read the corpus
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    text = f.read()

In [13]:
print("First 500 characters:\n")
print(text[:500])

First 500 characters:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [14]:
num_characters = len(text)
num_lines = len(text.splitlines())
num_words = len(text.split())

print(f"Characters : {num_characters:,}")
print(f"Lines      : {num_lines:,}")
print(f"Words      : {num_words:,}")

Characters : 1,115,393
Lines      : 40,000
Words      : 202,651


In [15]:
print(type(text))

<class 'str'>


In [16]:
#tokenization
# convert to lowercase
text = text.lower()
# tokenize
tokens = text.split()

In [17]:
print("First 20 tokens:\n")
print(tokens[:20])

First 20 tokens:

['first', 'citizen:', 'before', 'we', 'proceed', 'any', 'further,', 'hear', 'me', 'speak.', 'all:', 'speak,', 'speak.', 'first', 'citizen:', 'you', 'are', 'all', 'resolved', 'rather']


In [18]:
print(f"Total tokens      : {len(tokens):,}")
print(f"Unique tokens     : {len(set(tokens)):,}")

Total tokens      : 202,651
Unique tokens     : 23,641


In [19]:
print(type(tokens))
print(type(tokens[0]))

<class 'list'>
<class 'str'>


In [20]:
print(tokens[100:120])

['yield', 'us', 'but', 'the', 'superfluity,', 'while', 'it', 'were', 'wholesome,', 'we', 'might', 'guess', 'they', 'relieved', 'us', 'humanely;', 'but', 'they', 'think', 'we']


In [21]:
#build the vocabulary
word_counts = Counter(tokens)
print(word_counts.most_common(20))

[('the', 6279), ('and', 5479), ('to', 4723), ('i', 4403), ('of', 3721), ('my', 3114), ('a', 2975), ('you', 2449), ('that', 2427), ('in', 2312), ('is', 1963), ('for', 1835), ('with', 1800), ('not', 1741), ('your', 1680), ('be', 1597), ('his', 1521), ('he', 1411), ('as', 1404), ('but', 1402)]


In [22]:
vocabulary = [UNK_TOKEN]

for word, count in word_counts.items():
    if count >= MIN_FREQ:
        vocabulary.append(word)

In [23]:
print(f"Vocabulary Size: {len(vocabulary):,}")

Vocabulary Size: 23,642


In [24]:
word_to_idx = {
    word: idx
    for idx, word in enumerate(vocabulary)
}

In [25]:
print(word_to_idx["the"])
print(word_to_idx["king"])
print(word_to_idx["queen"])

30
6210
5596


In [26]:
idx_to_word = {
    idx: word
    for word, idx in word_to_idx.items()
}

In [27]:
idx = word_to_idx["king"]

print(idx)
print(idx_to_word[idx])

6210
king


In [28]:
print(f"Total Tokens      : {len(tokens):,}")
print(f"Unique Tokens     : {len(set(tokens)):,}")
print(f"Vocabulary Size   : {len(vocabulary):,}")
print(f"UNK Index         : {word_to_idx[UNK_TOKEN]}")

Total Tokens      : 202,651
Unique Tokens     : 23,641
Vocabulary Size   : 23,642
UNK Index         : 0


In [29]:
#numerical encoding
encoded_tokens = [
    word_to_idx.get(word, word_to_idx[UNK_TOKEN])
    for word in tokens
]

In [30]:
print("First 20 encoded tokens:\n")
print(encoded_tokens[:20])

First 20 encoded tokens:

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 10, 1, 2, 13, 14, 15, 16, 17]


In [31]:
for i in range(10):
    print(
        f"{tokens[i]:<15} -> {encoded_tokens[i]}"
    )

first           -> 1
citizen:        -> 2
before          -> 3
we              -> 4
proceed         -> 5
any             -> 6
further,        -> 7
hear            -> 8
me              -> 9
speak.          -> 10


In [32]:
for i in range(10):
    idx = encoded_tokens[i]
    print(
        f"{idx:<5} -> {idx_to_word[idx]}"
    )

1     -> first
2     -> citizen:
3     -> before
4     -> we
5     -> proceed
6     -> any
7     -> further,
8     -> hear
9     -> me
10    -> speak.


In [33]:
#store contexts and targets
contexts = []
targets = []

for i in range(len(encoded_tokens) - CONTEXT_SIZE):

    context = encoded_tokens[i : i + CONTEXT_SIZE]

    target = encoded_tokens[i + CONTEXT_SIZE]

    contexts.append(context)
    targets.append(target)

In [34]:
for i in range(5):
    print(f"Context: {contexts[i]} -> Target: {targets[i]}")

Context: [1, 2, 3] -> Target: 4
Context: [2, 3, 4] -> Target: 5
Context: [3, 4, 5] -> Target: 6
Context: [4, 5, 6] -> Target: 7
Context: [5, 6, 7] -> Target: 8


In [35]:
for i in range(5):

    context_words = [
        idx_to_word[idx]
        for idx in contexts[i]
    ]

    target_word = idx_to_word[targets[i]]

    print(context_words, "->", target_word)

['first', 'citizen:', 'before'] -> we
['citizen:', 'before', 'we'] -> proceed
['before', 'we', 'proceed'] -> any
['we', 'proceed', 'any'] -> further,
['proceed', 'any', 'further,'] -> hear


In [36]:
#Dataset and DataLoader
class NPLMDataset(Dataset):
    def __init__(self, contexts, targets):
        self.contexts = torch.tensor(contexts, dtype=torch.long)
        self.targets = torch.tensor(targets, dtype=torch.long)

    def __len__(self):
        return len(self.contexts)

    def __getitem__(self, idx):
        return self.contexts[idx], self.targets[idx]

In [37]:
dataset = NPLMDataset(contexts, targets)

In [38]:
train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [39]:
contexts_batch, targets_batch = next(iter(train_loader))

In [40]:
print(contexts_batch.shape)
print(targets_batch.shape)

torch.Size([64, 3])
torch.Size([64])


In [41]:
import torch.nn as nn

# Embedding matrix C
embedding = nn.Embedding(
    num_embeddings=len(word_to_idx),
    embedding_dim=EMBEDDING_DIM
)

print(embedding)

Embedding(23642, 50)


In [42]:
print("Embedding weight shape:", embedding.weight.shape)
print(embedding.weight[:5])

Embedding weight shape: torch.Size([23642, 50])
tensor([[ 6.7842e-01, -1.2345e+00, -4.3067e-02, -1.6047e+00,  1.7878e+00,
         -4.7805e-01, -2.4286e-01, -9.3416e-01, -7.2788e-01, -5.5943e-01,
         -7.6884e-01,  7.6245e-01, -1.5673e+00, -2.3945e-01,  2.3228e+00,
         -9.6337e-01, -7.5813e-01,  1.0783e+00,  8.0080e-01,  1.6806e+00,
          3.5586e-01, -6.8662e-01, -4.9336e-01,  2.4149e-01, -2.3162e-01,
          4.1759e-02, -2.5158e-01,  8.5986e-01, -3.0973e-01, -3.9571e-01,
          8.0341e-01, -6.2160e-01,  3.1888e-01, -4.2452e-01,  3.0572e-01,
         -7.7459e-01,  3.4912e-02,  3.2110e-01,  1.5736e+00, -8.4547e-01,
         -1.2742e+00,  2.1228e+00, -1.2347e+00, -4.8791e-01, -1.4181e+00,
          8.9627e-01,  4.9905e-02,  2.2667e+00, -4.8799e-01,  1.1914e+00],
        [-8.1401e-01, -7.3599e-01, -8.3712e-01, -9.2239e-01,  1.8113e+00,
          1.6056e-01, -9.7807e-02,  1.8446e+00, -1.1845e+00,  1.3835e+00,
         -1.2024e+00,  7.0781e-01, -1.0759e+00,  5.3565e-01,  3

In [43]:
# Get one mini-batch
contexts_batch, targets_batch = next(iter(train_loader))

# Retrieve embeddings
embedded = embedding(contexts_batch)

print("Context batch shape :", contexts_batch.shape)
print("Embedded shape      :", embedded.shape)

Context batch shape : torch.Size([64, 3])
Embedded shape      : torch.Size([64, 3, 50])


In [44]:
print("Context indices:")
print(contexts_batch[0])

print("\nEmbedding tensor shape:")
print(embedded[0].shape)

print("\nEmbedding vectors:")
print(embedded[0])

Context indices:
tensor([4353,   78,   59])

Embedding tensor shape:
torch.Size([3, 50])

Embedding vectors:
tensor([[ 0.8271, -0.5946,  1.3713,  0.2682,  0.4307,  0.6952,  0.7730,  0.5002,
         -0.2111,  0.1906, -0.9102,  0.1438, -0.6756, -0.7382, -0.1438, -1.1863,
         -0.6921,  0.8597, -0.3530, -0.2127, -0.7120, -0.0473, -0.7663,  0.9130,
          0.0463,  0.4252, -1.7158, -1.6098,  0.7255,  0.1046, -0.3923,  0.4285,
          0.6444,  0.4843, -1.2306,  0.0370, -0.1635,  1.9541,  1.4123,  0.5156,
          0.0885, -1.1491, -1.0682,  0.6204,  0.5312, -0.4056, -0.1389,  0.2424,
         -0.2629,  0.3469],
        [ 0.1483,  0.6686,  0.4213,  0.9768,  0.8450,  1.7977,  1.4310, -0.9844,
         -0.3203, -0.8570,  1.7586, -0.0940, -1.0033,  0.2110,  1.4641, -1.0836,
          1.1272, -0.5690, -0.7525, -0.6869,  0.4544,  0.1585, -0.0865,  0.2635,
         -0.6682, -0.3233, -0.3047, -0.2240,  2.9807, -0.8092, -0.7733, -1.0382,
         -0.7569,  0.7434, -1.0831,  2.4517, -0.6462,

In [45]:
# Verify the first context word of the first example
print(torch.allclose(
    embedded[0, 0],
    embedding.weight[contexts_batch[0, 0]]
))

True


In [46]:
# Concatenate embeddings into one feature vector
embedded_concat = embedded.view(embedded.size(0), -1)

print("Before:", embedded.shape)
print("After :", embedded_concat.shape)

Before: torch.Size([64, 3, 50])
After : torch.Size([64, 150])


In [47]:
print("Original shape:")
print(embedded[0].shape)

print("\nConcatenated shape:")
print(embedded_concat[0].shape)

Original shape:
torch.Size([3, 50])

Concatenated shape:
torch.Size([150])


In [48]:
# Hidden layer: a = Hx + d
hidden_layer = nn.Linear(
    in_features=CONTEXT_SIZE * EMBEDDING_DIM,
    out_features=HIDDEN_DIM
)

print(hidden_layer)

Linear(in_features=150, out_features=128, bias=True)


In [49]:
# Compute hidden layer pre-activation
hidden_pre_activation = hidden_layer(embedded_concat)

print("Input shape :", embedded_concat.shape)
print("Output shape:", hidden_pre_activation.shape)

Input shape : torch.Size([64, 150])
Output shape: torch.Size([64, 128])


In [50]:
print("Weight shape:", hidden_layer.weight.shape)
print("Bias shape  :", hidden_layer.bias.shape)

Weight shape: torch.Size([128, 150])
Bias shape  : torch.Size([128])


In [51]:
# Apply tanh activation
hidden = torch.tanh(hidden_pre_activation)

print("Before tanh:", hidden_pre_activation.shape)
print("After tanh :", hidden.shape)

Before tanh: torch.Size([64, 128])
After tanh : torch.Size([64, 128])


In [52]:
print("Pre-activation:")
print(hidden_pre_activation[0][:10])

print("\nAfter tanh:")
print(hidden[0][:10])

Pre-activation:
tensor([ 0.3339,  0.4855, -0.4060, -0.8946, -0.0462,  0.7496,  0.0465, -0.2807,
        -0.0435,  0.4639], grad_fn=<SliceBackward0>)

After tanh:
tensor([ 0.3221,  0.4506, -0.3851, -0.7137, -0.0461,  0.6349,  0.0465, -0.2735,
        -0.0435,  0.4332], grad_fn=<SliceBackward0>)


In [53]:
print("Minimum:", hidden.min().item())
print("Maximum:", hidden.max().item())

Minimum: -0.9776275157928467
Maximum: 0.9746854305267334


In [54]:
# Hidden-to-output transformation (Uh + b)
hidden_to_output = nn.Linear(
    in_features=HIDDEN_DIM,
    out_features=len(word_to_idx)
)

# Direct connection (Wx)
direct_connection = nn.Linear(
    in_features=CONTEXT_SIZE * EMBEDDING_DIM,
    out_features=len(word_to_idx),
    bias=False
)

In [55]:
hidden_scores = hidden_to_output(hidden)

direct_scores = direct_connection(embedded_concat)

logits = hidden_scores + direct_scores

In [56]:
print("Hidden scores :", hidden_scores.shape)
print("Direct scores :", direct_scores.shape)
print("Logits        :", logits.shape)

Hidden scores : torch.Size([64, 23642])
Direct scores : torch.Size([64, 23642])
Logits        : torch.Size([64, 23642])


In [57]:
probabilities = torch.softmax(logits, dim=1)

print("Logits shape       :", logits.shape)
print("Probability shape  :", probabilities.shape)

Logits shape       : torch.Size([64, 23642])
Probability shape  : torch.Size([64, 23642])


In [58]:
print(probabilities[0].sum())

tensor(1.0000, grad_fn=<SumBackward0>)


In [59]:
top_probs, top_indices = torch.topk(probabilities[0], k=5)

for prob, idx in zip(top_probs, top_indices):
    print(f"{idx.item():5d}  {idx_to_word[idx.item()]:15s}  {prob.item():.4f}")

10648  harmony:         0.0003
16241  answered.        0.0003
 8556  misconstrue      0.0003
13196  dun's            0.0003
18216  recall           0.0003


In [60]:
import torch
import torch.nn as nn


class NPLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        context_size,
        hidden_dim,
    ):
        super().__init__()

        # C
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
        )

        # H
        self.hidden_layer = nn.Linear(
            in_features=context_size * embedding_dim,
            out_features=hidden_dim,
        )

        # U (+ output bias)
        self.hidden_to_output = nn.Linear(
            in_features=hidden_dim,
            out_features=vocab_size,
        )

        # W (no bias)
        self.direct_connection = nn.Linear(
            in_features=context_size * embedding_dim,
            out_features=vocab_size,
            bias=False,
        )

    def forward(self, x):
        # Embedding lookup
        embedded = self.embedding(x)

        # Concatenate embeddings
        embedded = embedded.view(embedded.size(0), -1)

        # Hidden layer
        hidden = torch.tanh(
            self.hidden_layer(embedded)
        )

        # Output logits
        logits = (
            self.hidden_to_output(hidden)
            + self.direct_connection(embedded)
        )

        return logits

In [61]:
model = NPLM(
    vocab_size=len(word_to_idx),
    embedding_dim=EMBEDDING_DIM,
    context_size=CONTEXT_SIZE,
    hidden_dim=HIDDEN_DIM,
)

print(model)

NPLM(
  (embedding): Embedding(23642, 50)
  (hidden_layer): Linear(in_features=150, out_features=128, bias=True)
  (hidden_to_output): Linear(in_features=128, out_features=23642, bias=True)
  (direct_connection): Linear(in_features=150, out_features=23642, bias=False)
)


In [62]:
contexts_batch, targets_batch = next(iter(train_loader))

logits = model(contexts_batch)

print("Input shape :", contexts_batch.shape)
print("Logits shape:", logits.shape)

Input shape : torch.Size([64, 3])
Logits shape: torch.Size([64, 23642])


In [63]:
# Cross-entropy loss
criterion = nn.CrossEntropyLoss()

print(criterion)

CrossEntropyLoss()


In [64]:
contexts_batch, targets_batch = next(iter(train_loader))

logits = model(contexts_batch)

loss = criterion(logits, targets_batch)

print("Loss:", loss.item())

Loss: 10.341340065002441


In [65]:
print("Logits shape :", logits.shape)
print("Targets shape:", targets_batch.shape)

print("\nFirst target index:")
print(targets_batch[0])

Logits shape : torch.Size([64, 23642])
Targets shape: torch.Size([64])

First target index:
tensor(3801)


In [66]:
import torch.optim as optim

optimizer = optim.SGD(
    model.parameters(),
    lr=LEARNING_RATE
)

print(optimizer)

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.1
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)


In [67]:
params = list(model.parameters())

print("Number of parameter tensors:", len(params))

Number of parameter tensors: 6


In [68]:
for i, param in enumerate(model.parameters()):
    print(f"Parameter {i}: {param.shape}")

Parameter 0: torch.Size([23642, 50])
Parameter 1: torch.Size([128, 150])
Parameter 2: torch.Size([128])
Parameter 3: torch.Size([23642, 128])
Parameter 4: torch.Size([23642])
Parameter 5: torch.Size([23642, 150])


In [75]:
# Get one mini-batch
contexts_batch, targets_batch = next(iter(train_loader))

# Step 1: Clear old gradients
optimizer.zero_grad()

# Step 2: Forward pass
logits = model(contexts_batch)

# Step 3: Compute loss
loss = criterion(logits, targets_batch)

# Step 4: Compute gradients
loss.backward()

# Step 5: Update parameters
optimizer.step()

print(f"Loss: {loss.item():.4f}")

Loss: 10.2670


In [76]:
# Save a copy of one embedding vector
before = model.embedding.weight[0].clone()

# One training iteration
optimizer.zero_grad()

logits = model(contexts_batch)
loss = criterion(logits, targets_batch)

loss.backward()
optimizer.step()

after = model.embedding.weight[0]

print(torch.allclose(before, after))

True


In [77]:
# Pick the first word index from the first context
word_index = contexts_batch[0, 0].item()

before = model.embedding.weight[word_index].clone()

optimizer.zero_grad()

logits = model(contexts_batch)
loss = criterion(logits, targets_batch)

loss.backward()
optimizer.step()

after = model.embedding.weight[word_index]

print(f"Word index: {word_index}")
print("Embedding changed:", not torch.allclose(before, after))

Word index: 8
Embedding changed: True


In [78]:
model.train()

for epoch in range(EPOCHS):

    running_loss = 0.0

    for contexts_batch, targets_batch in train_loader:

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        logits = model(contexts_batch)

        # Compute loss
        loss = criterion(logits, targets_batch)

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()

        # Accumulate loss
        running_loss += loss.item()

    average_loss = running_loss / len(train_loader)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/20] Loss: 7.8513
Epoch [2/20] Loss: 7.0178
Epoch [3/20] Loss: 6.7711
Epoch [4/20] Loss: 6.5794
Epoch [5/20] Loss: 6.4119
Epoch [6/20] Loss: 6.2614
Epoch [7/20] Loss: 6.1217
Epoch [8/20] Loss: 5.9910
Epoch [9/20] Loss: 5.8670
Epoch [10/20] Loss: 5.7505
Epoch [11/20] Loss: 5.6378
Epoch [12/20] Loss: 5.5295
Epoch [13/20] Loss: 5.4261
Epoch [14/20] Loss: 5.3267
Epoch [15/20] Loss: 5.2302
Epoch [16/20] Loss: 5.1378
Epoch [17/20] Loss: 5.0490
Epoch [18/20] Loss: 4.9626
Epoch [19/20] Loss: 4.8793
Epoch [20/20] Loss: 4.7994


In [80]:
model.eval()

with torch.no_grad():

    contexts_batch, targets_batch = next(iter(train_loader))

    logits = model(contexts_batch)

    predictions = torch.argmax(logits, dim=1)

In [81]:
for i in range(5):

    predicted_word = idx_to_word[predictions[i].item()]
    actual_word = idx_to_word[targets_batch[i].item()]

    context_words = [
        idx_to_word[idx.item()]
        for idx in contexts_batch[i]
    ]

    print(f"Context   : {' '.join(context_words)}")
    print(f"Predicted : {predicted_word}")
    print(f"Actual    : {actual_word}")
    print("-" * 40)

Context   : arise to let
Predicted : him
Actual    : him
----------------------------------------
Context   : third--borough. sly: third,
Predicted : to
Actual    : or
----------------------------------------
Context   : that jars. how
Predicted : to
Actual    : fiery
----------------------------------------
Context   : your executioner, and
Predicted : that
Actual    : off
----------------------------------------
Context   : they can behold
Predicted : to
Actual    : bight
----------------------------------------


In [82]:
def generate_text(
    model,
    seed_text,
    num_words,
):
    model.eval()

    words = seed_text.lower().split()

    if len(words) != CONTEXT_SIZE:
        raise ValueError(
            f"Seed text must contain exactly {CONTEXT_SIZE} words."
        )

    context = [
        word_to_idx.get(
            word,
            word_to_idx[UNK_TOKEN]
        )
        for word in words
    ]

    generated = words.copy()

    with torch.no_grad():

        for _ in range(num_words):

            context_tensor = torch.tensor(
                [context],
                dtype=torch.long,
            )

            logits = model(context_tensor)

            next_word_idx = torch.argmax(
                logits,
                dim=1
            ).item()

            generated.append(
                idx_to_word[next_word_idx]
            )

            context = context[1:] + [next_word_idx]

    return " ".join(generated)

In [83]:
print(
    generate_text(
        model,
        seed_text="the king was",
        num_words=20,
    )
)

the king was the duke of norfolk, thomas mowbray, and he shall be the king of the ground, of the people in the


In [84]:
print(
    generate_text(
        model,
        seed_text="i will not",
        num_words=20,
    )
)

i will not be avoided to your princely presence. to the english king. the crown, of the king's ship i have a little


In [95]:
print(
    generate_text(
        model,
        seed_text="if i were",
        num_words=20,
    )
)

if i were a roman; of the king's ship i have a little of the whole of the people, of the people in


In [97]:
print(
    generate_text(
        model,
        seed_text="my good lord",
        num_words=20,
    )
)

my good lord of mercy, and that we may have been too much of mine own. and in the view of the people


In [100]:
print(
    generate_text(
        model,
        seed_text="would you have",
        num_words=20,
    )
)

would you have been a woman of the king's ship i have a little of the whole of the people, of the people
